# OLD VERSION

In [5]:
import pandas as pd
import lightkurve as lk
import time

In [6]:
# load data from .csv
table = pd.read_csv("TESS_M=0.4_RP=15.csv")

In [7]:
# for now do 20 because all would take long time
subset = table.head(20)

In [8]:
rows = []

In [9]:
# loops .csv
for id, ra, dec in zip(subset['source_id'], subset['ra'], subset['dec']):
    print(f"Checking source_id {id} (RA={ra}, Dec={dec})")
    try:
        # goes through TESS Full Frame Image (FFI)
        search = lk.search_tesscut('%s %s' % (ra, dec))
        for values in search:
            tab = values.table

            # get sector (either sequence_number/sector)
            if "sequence_number" in tab.colnames:
                sector = tab["sequence_number"][0]
            elif "sector" in tab.colnames:
                sector = tab["sector"][0]
            #else:
              #  sector = None

            # get exposure time
            exptime = tab["exptime"][0] 
            
            # append
            rows.append({
                "ID": id,
                "Sector": sector,
                "Exposure_Time": exptime
            })
    except Exception as e:
        print(f"Error for {id}: {e}")

Checking source_id 31958852451968 (RA=44.855783116029166, Dec=0.4211441654503658)
Checking source_id 77863462880768 (RA=44.660730046322826, Dec=0.5174706461824936)
Checking source_id 97551592960640 (RA=44.71163306521175, Dec=0.6747353569305051)
Checking source_id 110951890948096 (RA=44.96551311298629, Dec=0.771972846155539)
Checking source_id 115899693267968 (RA=45.1410279662417, Dec=0.8430052599114634)
Checking source_id 123424475948672 (RA=44.8000069221394, Dec=0.7794933038508934)
Checking source_id 154829276705920 (RA=45.79773407480096, Dec=0.9005394643362447)
Checking source_id 255709468513920 (RA=45.91261574351641, Dec=1.4039870019524443)
Checking source_id 290653322568704 (RA=44.44850159775912, Dec=0.7917886784873427)
Checking source_id 319545567090560 (RA=44.7927374760415, Dec=1.034519863807558)
Checking source_id 339993906853248 (RA=44.36039043546913, Dec=1.2086815246994866)
Checking source_id 435754497122304 (RA=45.051993502946864, Dec=1.4570431652076004)
Checking source_id 45

In [10]:
out = pd.DataFrame(rows)
display(out)

,ID,Sector,Exposure_Time
0,31958852451968,4,1425.599419
1,31958852451968,31,475.199790
2,77863462880768,4,1425.599419
3,97551592960640,4,1425.599419
4,97551592960640,31,475.199790
5,110951890948096,4,1425.599419
6,110951890948096,31,475.199790
7,115899693267968,4,1425.599419
8,115899693267968,31,475.199790
9,123424475948672,4,1425.599419


In [11]:
#save spreadsheet .csv
out.to_csv("tess_id_sector_exptime_first20.csv")
print("Saved in tess_id_sector_exptime_first20.csv")

Saved in tess_id_sector_exptime_first20.csv


# NEWEST VERSION

In [2]:
import pandas as pd
import lightkurve as lk
import time

In [3]:
# load data from .csv
table = pd.read_csv("TESS_M=0.4_RP=15.csv")

# convert parallax from mas to pc
table['distance_pc'] = 1000 / table['parallax']  

# filtered to have limit of G mag <16 and d<20
filtered = table[
    (table['phot_g_mean_mag'] <= 16) &
    (table['distance_pc'] <= 20)
]

# tells me total targets before and after filtering
print("Total sources before filtering:", len(table))
print("Sources after filtering:", len(filtered))
#print(filtered[['source_id','phot_g_mean_mag','distance_pc']].head())

Total sources before filtering: 140813
Sources after filtering: 736


In [4]:
subset = filtered

rows = []

# loops .csv
for i, (id, ra, dec) in enumerate(zip(subset['source_id'], subset['ra'], subset['dec']), start=1):
    print(f"{i}. Checking source_id {id} (RA={ra}, DEC={dec})")
    try:
        # goes through TESS Full Frame Image (FFI)
        search = lk.search_tesscut('%s %s' % (ra, dec))

        if len(search) == 0:
            # No FFI found, will tell me it's false
            rows.append({
                "ID": id,
                "Sector": None,
                "Exposure_Time": None,
                "FFI": False
            })
            continue

        for values in search:
            tab = values.table

            # get sector (either sequence_number/sector)
            if "sequence_number" in tab.colnames:
                sector = tab["sequence_number"][0]
            elif "sector" in tab.colnames:
                sector = tab["sector"][0]

            # get exposure time
            exptime = tab["exptime"][0] 

            # append (now with FFI column)
            rows.append({
                "ID": id,
                "Sector": sector,
                "Exposure_Time": exptime,
                "FFI": True    # 
            })

    except Exception as e:
        print(f"Error for {id}: {e}")
        rows.append({
            "ID": id,
            "Sector": None,
            "Exposure_Time": None,
            "FFI": False
        })

out = pd.DataFrame(rows)
display(out)

1. Checking source_id 769456276704128 (RA=47.51489911594037, DEC=2.572769511017299)
2. Checking source_id 16302463200535040 (RA=50.8422709188759, DEC=11.686387390947743)
3. Checking source_id 18986817760556416 (RA=39.82435828918509, DEC=7.470816071340625)
4. Checking source_id 35398295820372864 (RA=43.358915254728714, DEC=17.407883277761023)
5. Checking source_id 57739208163471744 (RA=51.68738385482385, DEC=19.243802626944703)


KeyboardInterrupt: 

In [10]:
#save spreadsheet .csv
out.to_csv("tess_id_sector_exptime_G16d20_all_filtered.csv", index=False)
print("Saved in tess_id_sector_exptime_G16d20_all_filtered.csv")


Saved in tess_id_sector_exptime_G16d20_all_filtered.csv


## Will determine FFI based on ID

In [28]:
import pandas as pd
import lightkurve as lk

table = pd.read_csv("TESS_M=0.4_RP=15.csv")

id = 4349305645979265920


# lookup RA and Dec
row = table[table['source_id'] == id].iloc[0]
ra  = row['ra']
dec = row['dec']

search = lk.search_tesscut("%s %s" % (ra, dec))

if len(search) > 0:
    print(f"ID {id}: FFI: YES")
    print("Number of sectors:", len(search))
    print(search)
else:
    print(f"ID {id}: FFI: NO")


ID 4349305645979265920: FFI: NO


## Will determine FFI based on RA + DEC

In [22]:
import lightkurve as lk 

# test if ID has FFI 
# case where it has it 
#ra = 269.59373567871177 
#dec = -12.760552335950688 

#case where it doesn't 
ra = 265.4751620316947 
dec = 9.680499876229684 

search = lk.search_tesscut("%s %s" % (ra, dec)) 
if len(search) > 0: 
    print("FFI coverage: YES") 
    print("Number of sectors:", len(search)) 
    print(search) 
else: 
    print("FFI coverage: NO")

FFI coverage: NO


## Spreadsheet now impliments SPOC availability
* search_tesscut - finds FFI
* search_targetpixelfile - finds SPOC in short cadence target pixel files


In [1]:
import pandas as pd
import lightkurve as lk
import time

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/lightkurve/prf/__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


In [17]:
# load data from .csv
table = pd.read_csv("TESS_M=0.4_RP=15.csv")

# convert parallax from mas to pc
table['distance_pc'] = 1000 / table['parallax']  

# filtered to have limit of G mag <16 and d<20
filtered = table[
    (table['phot_g_mean_mag'] <= 16) &
    (table['distance_pc'] <= 20)
]

# tells me total targets before and after filtering
print("Total sources before filtering:", len(table))
print("Sources after filtering:", len(filtered))
#print(filtered[['source_id','phot_g_mean_mag','distance_pc']].head())

subset = filtered.head(5)
#subset = filtered
rows = []

Total sources before filtering: 140813
Sources after filtering: 736


In [18]:
#UPDATE
def unwrap_scalar(x):
    """Convert 1-element array/Series/etc. into a scalar for cleaner CSV display."""
    if x is None:
        return None
    if isinstance(x, (str, bytes)):
        return x
    try:
        if hasattr(x, "__len__") and len(x) == 1:
            return x[0]
    except Exception:
        pass
    return x

# loops .csv
for i, (id, ra, dec) in enumerate(zip(subset['source_id'], subset['ra'], subset['dec']), start=1):
    print(f"{i}. Checking source_id {id} (RA={ra}, DEC={dec})")
    try:
        coord = f"{ra} {dec}"

        # ----------------------------
        # FFI (TESSCut) search
        # ----------------------------
        search_ffi = lk.search_tesscut(coord)
        ffi_avail = len(search_ffi) > 0

        # ----------------------------
        # SPOC (short cadence) search
        # ----------------------------
        spoc = lk.search_targetpixelfile(f"Gaia DR3 {id}", mission="TESS", author="SPOC")
        spoc_avail = len(spoc) > 0

        # If no FFI AND no SPOC, still record a row
        if (not ffi_avail) and (not spoc_avail):
            rows.append({
                "ID": id,
                "DataType": None,
                "Sector": None,
                "Exposure_Time": None,
                "FFI_Avail": False,
                "SPOC_Avail": False
            })
            continue

        # ----------------------------
        # Add rows for FFI products
        # ----------------------------
        if ffi_avail:
            for values in search_ffi:
                tab = values.table

                # sector (either sequence_number/sector)
                sector = None
                if "sequence_number" in tab.colnames:
                    sector = tab["sequence_number"][0]
                elif "sector" in tab.colnames:
                    sector = tab["sector"][0]

                # exposure time
                exptime = tab["exptime"][0] if "exptime" in tab.colnames else None

                sector = unwrap_scalar(sector)
                exptime = unwrap_scalar(exptime)

                rows.append({
                    "ID": id,
                    "DataType": "FFI",
                    "Sector": sector,
                    "Exposure_Time": exptime,
                    "FFI_Avail": True,
                    "SPOC_Avail": spoc_avail
                })
        else:
            # No FFI found
            rows.append({
                "ID": id,
                "DataType": "FFI",
                "Sector": None,
                "Exposure_Time": None,
                "FFI_Avail": False,
                "SPOC_Avail": spoc_avail
            })

        # ----------------------------
        # Add rows for SPOC products
        # ----------------------------
        if spoc_avail:
            for values in spoc:
                sector = getattr(values, "sector", None)
                exptime = getattr(values, "exptime", None)

                # Fallback to table columns if needed
                if sector is None or exptime is None:
                    tab = values.table
                    if sector is None and "sector" in tab.colnames:
                        sector = tab["sector"][0]
                    if exptime is None and "exptime" in tab.colnames:
                        exptime = tab["exptime"][0]

                sector = unwrap_scalar(sector)
                exptime = unwrap_scalar(exptime)

                rows.append({
                    "ID": id,
                    "DataType": "SPOC",
                    "Sector": sector,
                    "Exposure_Time": exptime,
                    "FFI_Avail": ffi_avail,
                    "SPOC_Avail": True
                })
        else:
            # No SPOC found
            rows.append({
                "ID": id,
                "DataType": "SPOC",
                "Sector": None,
                "Exposure_Time": None,
                "FFI_Avail": ffi_avail,
                "SPOC_Avail": False
            })

    except Exception as e:
        print(f"Error for {id}: {e}")
        rows.append({
            "ID": id,
            "DataType": None,
            "Sector": None,
            "Exposure_Time": None,
            "FFI_Avail": False,
            "SPOC_Avail": False
        })

out = pd.DataFrame(rows)
display(out)

# save spreadsheet .csv
#out.to_csv("tess_id_sector_exptime_G16d20_all_filtered.csv", index=False)
#print("Saved in tess_id_sector_exptime_G16d20_all_filtered.csv")


1. Checking source_id 769456276704128 (RA=47.51489911594037, DEC=2.572769511017299)
2. Checking source_id 16302463200535040 (RA=50.8422709188759, DEC=11.686387390947743)
3. Checking source_id 18986817760556416 (RA=39.82435828918509, DEC=7.470816071340625)
4. Checking source_id 35398295820372864 (RA=43.358915254728714, DEC=17.407883277761023)
5. Checking source_id 57739208163471744 (RA=51.68738385482385, DEC=19.243802626944703)


,ID,DataType,Sector,Exposure_Time,FFI_Avail,SPOC_Avail
0,769456276704128,FFI,4.0,1425.599419,True,True
1,769456276704128,FFI,31.0,475.19979,True,True
2,769456276704128,SPOC,NaN,120.0 s,True,True
3,769456276704128,SPOC,NaN,120.0 s,True,True
4,16302463200535040,FFI,31.0,475.19979,True,True
5,16302463200535040,FFI,42.0,475.199778,True,True
6,16302463200535040,FFI,43.0,475.19979,True,True
7,16302463200535040,FFI,44.0,475.19979,True,True
8,16302463200535040,FFI,70.0,158.399927,True,True
9,16302463200535040,FFI,71.0,158.39993,True,True


In [7]:
# loops .csv
for i, (id, ra, dec) in enumerate(zip(subset['source_id'], subset['ra'], subset['dec']), start=1):
    print(f"{i}. Checking source_id {id} (RA={ra}, DEC={dec})")
    try:
        # goes through TESS Full Frame Image (FFI)
        search = lk.search_tesscut('%s %s' % (ra, dec))

        # see if SPOC is available
        # SPOC pipeline used to process all short cadence data
        spoc = lk.search_targetpixelfile('%s %s' % (ra, dec), mission="TESS", author="SPOC")
        spoc_avail = len(spoc) > 0
        

        if len(search) == 0:
            # No FFI found
            rows.append({
                "ID": id,
                "Sector": None,
                "Exposure_Time": None,
                "FFI": False,
                "SPOC": spoc_avail
            })
            continue

        for values in search:
            tab = values.table

            # get sector (either sequence_number/sector)
            if "sequence_number" in tab.colnames:
                sector = tab["sequence_number"][0]
            elif "sector" in tab.colnames:
                sector = tab["sector"][0]

            # get exposure time
            exptime = tab["exptime"][0] 

            # append
            rows.append({
                "ID": id,
                "Sector": sector,
                "Exposure_Time": exptime,
                "FFI": True,
                "SPOC": spoc_avail
            })

    except Exception as e:
        print(f"Error for {id}: {e}")
        rows.append({
            "ID": id,
            "Sector": None,
            "Exposure_Time": None,
            "FFI": False,
            "SPOC": False
        })

out = pd.DataFrame(rows)
display(out)

#save spreadsheet .csv
out.to_csv("tess_id_sector_exptime_G16d20_all_filtered.csv", index=False)
print("Saved in tess_id_sector_exptime_G16d20_all_filtered.csv")


1. Checking source_id 769456276704128 (RA=47.51489911594037, DEC=2.572769511017299)


No data found for target "47.51489911594037 2.572769511017299".


2. Checking source_id 16302463200535040 (RA=50.8422709188759, DEC=11.686387390947743)
3. Checking source_id 18986817760556416 (RA=39.82435828918509, DEC=7.470816071340625)


No data found for target "39.82435828918509 7.470816071340625".


4. Checking source_id 35398295820372864 (RA=43.358915254728714, DEC=17.407883277761023)


...

733. Checking source_id 6143175943486753152 (RA=182.48992008073932, DEC=-46.19264116008711)


No data found for target "182.48992008073932 -46.19264116008711".


734. Checking source_id 6725832095922016256 (RA=273.3347119890693, DEC=-39.84395218207264)
735. Checking source_id 6734485286788609664 (RA=277.3507091117809, DEC=-34.96362316023356)
736. Checking source_id 6759481141756109056 (RA=290.376709235217, DEC=-29.2629248637208)


,ID,Sector,Exposure_Time,FFI,SPOC
0,769456276704128,4.0,1425.599419,True,False
1,769456276704128,31.0,475.199790,True,False
2,16302463200535040,31.0,475.199790,True,True
3,16302463200535040,42.0,475.199778,True,True
4,16302463200535040,43.0,475.199790,True,True
...,...,...,...,...,...
4263,6734485286788609664,93.0,158.399924,True,True
4264,6734485286788609664,92.0,158.399926,True,True
4265,6759481141756109056,13.0,1425.599392,True,True
4266,6759481141756109056,27.0,475.199782,True,True


Saved in tess_id_sector_exptime_G16d20_all_filtered.csv


## Code that verifies SPOC availbility as sanity check

In [16]:
import pandas as pd
import lightkurve as lk

# load original Gaia data and output spreadsheet
table = pd.read_csv("TESS_M=0.4_RP=15.csv")
out = pd.read_csv("tess_id_sector_exptime_G16d20_all_filtered.csv")

# -----------------------------
# manually choose row by index:
i = 2  # <-- change this index to inspect different stars
# -----------------------------

# get spreadsheet row
row = out.iloc[i]

# extract its Gaia ID
gaia_id = row["ID"]
print(f"\nID {gaia_id} (from spreadsheet SPOC={row['SPOC']})")

# getting RA/DEC from original .csv
orig = table[table["source_id"] == gaia_id].iloc[0]
ra, dec = orig["ra"], orig["dec"]
print(f"RA={ra}, DEC={dec}")

# query Lightkurve for SPOC
search = lk.search_targetpixelfile(f"{ra} {dec}", mission="TESS", author="SPOC")
print(search)

if len(search) > 0:
    tpf = search.download()
    print("\nDownloaded:")
    print(tpf)
else:
    print("\nNo SPOC TPF available for this target.")



ID 16302463200535040 (from spreadsheet SPOC=True)
RA=50.8422709188759, DEC=11.686387390947743
SearchResult containing 5 data products.

 #     mission     year author exptime target_name distance
                                  s                 arcsec 
--- -------------- ---- ------ ------- ----------- --------
  0 TESS Sector 31 2020   SPOC      20   381133463      0.0
  1 TESS Sector 31 2020   SPOC     120   381133463      0.0
  2 TESS Sector 42 2021   SPOC     120   381133463      0.0
  3 TESS Sector 43 2021   SPOC     120   381133463      0.0
  4 TESS Sector 44 2021   SPOC     120   381133463      0.0

Downloaded:
TessTargetPixelFile(TICID: 381133463)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/lightkurve/search.py:420: LightkurveWarning: Warning: 5 files available to download. Only the first file has been downloaded. Please use `download_all()` or specify additional criteria (e.g. quarter, campaign, or sector) to limit your search.
  warnings.warn(


In [17]:
import lightkurve as lk

search = lk.search_targetpixelfile("Gaia Dr3 16302463200535040", mission="TESS")
search
tpf = search.download()
tpf.plot();


HTTPError: 500 Server Error: Internal Server Error for url: https://mast.stsci.edu/api/v0/invoke?request=%7B%22service%22%3A%20%22Mast.Name.Lookup%22%2C%20%22params%22%3A%20%7B%22input%22%3A%20%22Gaia%20Dr3%2016302463200535040%22%2C%20%22format%22%3A%20%22json%22%7D%7D

In [15]:
lk.search_targetpixelfile("Gaia DR3 769456276704128", mission="TESS", author="SPOC")


HTTPError: 500 Server Error: Internal Server Error for url: https://mast.stsci.edu/api/v0/invoke?request=%7B%22service%22%3A%20%22Mast.Name.Lookup%22%2C%20%22params%22%3A%20%7B%22input%22%3A%20%22Gaia%20DR3%20769456276704128%22%2C%20%22format%22%3A%20%22json%22%7D%7D

In [20]:
import pandas as pd
import lightkurve as lk
import time

In [21]:
# load data from .csv
table = pd.read_csv("TESS_M=0.4_RP=15.csv")

# convert parallax from mas to pc
table['distance_pc'] = 1000 / table['parallax']  

# filtered to have limit of G mag <16 and d<20
filtered = table[
    (table['phot_g_mean_mag'] <= 16) &
    (table['distance_pc'] <= 20)
]

# tells me total targets before and after filtering
print("Total sources before filtering:", len(table))
print("Sources after filtering:", len(filtered))
#print(filtered[['source_id','phot_g_mean_mag','distance_pc']].head())

#subset = filtered.head(5)
subset = filtered
rows = []

Total sources before filtering: 140813
Sources after filtering: 736


In [22]:
# loops .csv
for i, (id, ra, dec) in enumerate(zip(subset['source_id'], subset['ra'], subset['dec']), start=1):
    print(f"{i}. Checking source_id {id} (RA={ra}, DEC={dec})")

    try:
        coord = f"{ra} {dec}"

        # Looks specifically for FFI (TESSCut)
        search_ffi = lk.search_tesscut(coord)
        ffi_avail = len(search_ffi) > 0

        # Looks specifically for SPOC (short cadence)
        spoc = lk.search_targetpixelfile(
            f"Gaia DR3 {id}",
            mission="TESS",
            author="SPOC"
        )
        spoc_avail = len(spoc) > 0

        # If it doesn't have either of them, it will still record
        if not ffi_avail and not spoc_avail:
            rows.append({
                "ID": id,
                "DataType": None,
                "Sector": None,
                "Exposure_Time": None,
                "FFI_Avail": False,
                "SPOC_Avail": False
            })
            continue

        # Adds the rows for FFI
        if ffi_avail:
            for values in search_ffi:
                tab = values.table

                sector = None
                if "sequence_number" in tab.colnames:
                    sector = tab["sequence_number"][0]
                elif "sector" in tab.colnames:
                    sector = tab["sector"][0]

                exptime = tab["exptime"][0] if "exptime" in tab.colnames else None

                rows.append({
                    "ID": id,
                    "DataType": "FFI",
                    "Sector": sector,
                    "Exposure_Time": exptime,
                    "FFI_Avail": True,
                    "SPOC_Avail": spoc_avail
                })

        # Adds the rows for SPOC
        if spoc_avail:
            for values in spoc:
                sector = values.sector if hasattr(values, "sector") else None
                exptime = values.exptime if hasattr(values, "exptime") else None

                # Fallback to table columns if needed
                if sector is None or exptime is None:
                    tab = values.table
                    if sector is None and "sector" in tab.colnames:
                        sector = tab["sector"][0]
                    if exptime is None and "exptime" in tab.colnames:
                        exptime = tab["exptime"][0]

                rows.append({
                    "ID": id,
                    "DataType": "SPOC",
                    "Sector": sector,
                    "Exposure_Time": exptime,
                    "FFI_Avail": ffi_avail,
                    "SPOC_Avail": True
                })

    except Exception as e:
        print(f"Error for {id}: {e}")
        rows.append({
            "ID": id,
            "DataType": None,
            "Sector": None,
            "Exposure_Time": None,
            "FFI_Avail": False,
            "SPOC_Avail": False
        })
 #if i % 25 == 0:
  #      pd.DataFrame(rows).to_csv(
   #         "tess_id_sector_exptime_G16d20_all_filtered_UPDATED_PARTIAL.csv",
    #        index=False
     #   )
      #  print(f"Checkpoint saved at object {i}")
 
out = pd.DataFrame(rows)
display(out)

# save spreadsheet .csv
out.to_csv("tess_id_sector_exptime_G16d20_all_filtered_UPDATED.csv", index=False)
print("Saved in tess_id_sector_exptime_G16d20_all_filtered_UPDATED.csv")


1. Checking source_id 769456276704128 (RA=47.51489911594037, DEC=2.572769511017299)
2. Checking source_id 16302463200535040 (RA=50.8422709188759, DEC=11.686387390947743)
3. Checking source_id 18986817760556416 (RA=39.82435828918509, DEC=7.470816071340625)
4. Checking source_id 35398295820372864 (RA=43.358915254728714, DEC=17.407883277761023)


No data found for target "Gaia DR3 35398295820372864".


5. Checking source_id 57739208163471744 (RA=51.68738385482385, DEC=19.243802626944703)
6. Checking source_id 68553931516671488 (RA=54.87437762176911, DEC=24.96824115907829)
7. Checking source_id 72117556076986368 (RA=33.98650746472387, DEC=10.254962202598763)
8. Checking source_id 72354810070468224 (RA=33.192827917024836, DEC=10.547975698321418)
9. Checking source_id 89593331327905536 (RA=39.183682325426595, DEC=22.67227965316612)
10. Checking source_id 91196316201965184 (RA=29.56508818927173, DEC=18.12063665648874)
11. Checking source_id 97334889619799168 (RA=27.85070481391559, DEC=21.39275062405243)
12. Checking source_id 97993497084921984 (RA=27.018206566174985, DEC=21.205767467992107)
13. Checking source_id 117709729140217216 (RA=52.20764798129931, DEC=26.48616945277209)
14. Checking source_id 308828253324985984 (RA=19.459213398032063, DEC=28.669329862628036)
15. Checking source_id 327944328126649856 (RA=34.29473704167926, DEC=35.44121898612927)
16. Checking source_id 3480202422174

No data found for target "Gaia DR3 348020242217448576".


17. Checking source_id 373838214051410816 (RA=14.504225894844094, DEC=39.31987471822514)
18. Checking source_id 569821974111727744 (RA=63.51645882603903, DEC=82.25759093058488)
19. Checking source_id 598180646733241216 (RA=130.7186251598315, DEC=9.55037430644196)
20. Checking source_id 605079910398496640 (RA=134.08122584963937, DEC=12.662731415781227)
21. Checking source_id 615779464207025152 (RA=150.6775798896445, DEC=14.985896812309678)
22. Checking source_id 617982090939470464 (RA=144.87402457514528, DEC=14.646793438572836)
23. Checking source_id 619440661833351040 (RA=147.20907756971008, DEC=15.646815201966763)
24. Checking source_id 622339455521105280 (RA=153.97610368902318, DEC=17.490880663782907)
25. Checking source_id 628080246633611520 (RA=148.4782397121578, DEC=20.948079178895128)
26. Checking source_id 628865130046268672 (RA=153.72046302348045, DEC=21.395135245402663)
27. Checking source_id 641821259671808896 (RA=149.11007978867167, DEC=22.64914689456202)
28. Checking source

...

673. Checking source_id 5418743844032438400 (RA=150.12849579829455, DEC=-41.50815733346062)
674. Checking source_id 5424690587034982144 (RA=143.35813068013894, DEC=-43.89273302384089)
675. Checking source_id 5432670704985289088 (RA=145.8742485909251, DEC=-38.56630443256017)
676. Checking source_id 5439029627401396096 (RA=146.491875412928, DEC=-32.89101863068049)
677. Checking source_id 5440359486716251904 (RA=157.78388333811384, DEC=-41.46334165948587)
678. Checking source_id 5441169002153568512 (RA=159.8833472486551, DEC=-38.3370450973924)
679. Checking source_id 5458784415381054464 (RA=151.1632016429737, DEC=-33.58751841752631)
680. Checking source_id 5492622847798742272 (RA=114.05296644815438, DEC=-51.920572905284935)
681. Checking source_id 5519117229744210944 (RA=119.2234839106459, DEC=-45.63440964675406)
682. Checking source_id 5529120822749728384 (RA=130.17026521119897, DEC=-38.5440354728072)
683. Checking source_id 5538156819768871552 (RA=118.72808136767864, DEC=-38.15886028136

No data found for target "Gaia DR3 6143175943486753152".


734. Checking source_id 6725832095922016256 (RA=273.3347119890693, DEC=-39.84395218207264)
735. Checking source_id 6734485286788609664 (RA=277.3507091117809, DEC=-34.96362316023356)
736. Checking source_id 6759481141756109056 (RA=290.376709235217, DEC=-29.2629248637208)


,ID,DataType,Sector,Exposure_Time,FFI_Avail,SPOC_Avail
0,769456276704128,FFI,4.0,1425.599419,True,True
1,769456276704128,FFI,31.0,475.19979,True,True
2,769456276704128,SPOC,NaN,[120.0 s],True,True
3,769456276704128,SPOC,NaN,[120.0 s],True,True
4,16302463200535040,FFI,31.0,475.19979,True,True
...,...,...,...,...,...,...
8950,6734485286788609664,SPOC,NaN,[120.0 s],True,True
8951,6759481141756109056,FFI,13.0,1425.599392,True,True
8952,6759481141756109056,FFI,27.0,475.199782,True,True
8953,6759481141756109056,FFI,92.0,158.399926,True,True


Saved in tess_id_sector_exptime_G16d20_all_filtered_UPDATED_test.csv


In [14]:
#looks specifically for spoc (short cadence, selected targets)
lk.search_targetpixelfile("Gaia DR3 769456276704128", mission="TESS", author="SPOC")

HTTPError: 500 Server Error: Internal Server Error for url: https://mast.stsci.edu/api/v0/invoke?request=%7B%22service%22%3A%20%22Mast.Name.Lookup%22%2C%20%22params%22%3A%20%7B%22input%22%3A%20%22Gaia%20DR3%20769456276704128%22%2C%20%22format%22%3A%20%22json%22%7D%7D

In [19]:
#FFI
lk.search_tesscut("Gaia DR3 769456276704128")

#,mission,year,author,exptime,target_name,distance
,,,,s,,arcsec
0,TESS Sector 04,2018,TESScut,1426,Gaia DR3 769456276704128,0.0
1,TESS Sector 31,2020,TESScut,475,Gaia DR3 769456276704128,0.0


In [25]:
import pandas as pd
import lightkurve as lk
import time
import re

def get_sector_from_tab(tab):
    """Try to read sector from columns; otherwise parse from mission string 'TESS Sector XX'."""
    # direct columns first
    if "sector" in tab.colnames:
        try:
            return int(tab["sector"][0])
        except Exception:
            pass
    if "sequence_number" in tab.colnames:
        try:
            return int(tab["sequence_number"][0])
        except Exception:
            pass

    # parse from mission string (common for search_targetpixelfile results)
    if "mission" in tab.colnames:
        m = re.search(r"Sector\s+(\d+)", str(tab["mission"][0]))
        if m:
            return int(m.group(1))

    return None


# load data from .csv
table = pd.read_csv("TESS_M=0.4_RP=15.csv")

# convert parallax from mas to pc
table["distance_pc"] = 1000 / table["parallax"]

# filtered to have limit of G mag <16 and d<20
filtered = table[
    (table["phot_g_mean_mag"] <= 16) &
    (table["distance_pc"] <= 20)
]

print("Total sources before filtering:", len(table))
print("Sources after filtering:", len(filtered))

#subset = filtered.head(5)
subset = filtered
rows = []

for i, (id, ra, dec) in enumerate(zip(subset["source_id"], subset["ra"], subset["dec"]), start=1):
    print(f"{i}. Checking source_id {id} (RA={ra}, DEC={dec})")

    try:
        coord = f"{ra} {dec}"

        # Looks specifically for FFI (TESSCut)
        search_ffi = lk.search_tesscut(coord)
        ffi_avail = len(search_ffi) > 0

        # Looks specifically for SPOC (target pixel files)
        spoc = lk.search_targetpixelfile(
            f"Gaia DR3 {id}",
            mission="TESS",
            author="SPOC"
        )
        spoc_avail = len(spoc) > 0

        if not ffi_avail and not spoc_avail:
            rows.append({
                "ID": id,
                "DataType": None,
                "Sector": None,
                "Exposure_Time": None,
                "FFI_Avail": False,
                "SPOC_Avail": False
            })
            continue

        # Adds the rows for FFI
        if ffi_avail:
            for values in search_ffi:
                tab = values.table

                sector = None
                if "sequence_number" in tab.colnames:
                    sector = tab["sequence_number"][0]
                elif "sector" in tab.colnames:
                    sector = tab["sector"][0]

                exptime = tab["exptime"][0] if "exptime" in tab.colnames else None

                rows.append({
                    "ID": id,
                    "DataType": "FFI",
                    "Sector": sector,
                    "Exposure_Time": exptime,
                    "FFI_Avail": True,
                    "SPOC_Avail": spoc_avail
                })

        # Adds the rows for SPOC
        if spoc_avail:
            for values in spoc:
                tab = values.table

                sector = get_sector_from_tab(tab)

                # exptime should be in the table for search_targetpixelfile results
                exptime = tab["exptime"][0] if "exptime" in tab.colnames else None

                rows.append({
                    "ID": id,
                    "DataType": "SPOC",
                    "Sector": sector,
                    "Exposure_Time": exptime,
                    "FFI_Avail": ffi_avail,
                    "SPOC_Avail": True
                })

    except Exception as e:
        print(f"Error for {id}: {e}")
        rows.append({
            "ID": id,
            "DataType": None,
            "Sector": None,
            "Exposure_Time": None,
            "FFI_Avail": False,
            "SPOC_Avail": False
        })

out = pd.DataFrame(rows)
display(out)

out.to_csv("tess_id_sector_exptime_G16d20_all_filtered_UPDATED_SEC.csv", index=False)
print("Saved in tess_id_sector_exptime_G16d20_all_filtered_UPDATED_SEC.csv")


Total sources before filtering: 140813
Sources after filtering: 736
1. Checking source_id 769456276704128 (RA=47.51489911594037, DEC=2.572769511017299)
2. Checking source_id 16302463200535040 (RA=50.8422709188759, DEC=11.686387390947743)
3. Checking source_id 18986817760556416 (RA=39.82435828918509, DEC=7.470816071340625)
4. Checking source_id 35398295820372864 (RA=43.358915254728714, DEC=17.407883277761023)
5. Checking source_id 57739208163471744 (RA=51.68738385482385, DEC=19.243802626944703)
6. Checking source_id 68553931516671488 (RA=54.87437762176911, DEC=24.96824115907829)
7. Checking source_id 72117556076986368 (RA=33.98650746472387, DEC=10.254962202598763)
8. Checking source_id 72354810070468224 (RA=33.192827917024836, DEC=10.547975698321418)
9. Checking source_id 89593331327905536 (RA=39.183682325426595, DEC=22.67227965316612)
10. Checking source_id 91196316201965184 (RA=29.56508818927173, DEC=18.12063665648874)
11. Checking source_id 97334889619799168 (RA=27.85070481391559, DE

,ID,DataType,Sector,Exposure_Time,FFI_Avail,SPOC_Avail
0,769456276704128,FFI,4.0,1425.599419,True,True
1,769456276704128,FFI,31.0,475.199790,True,True
2,769456276704128,SPOC,4.0,120.000000,True,True
3,769456276704128,SPOC,31.0,120.000000,True,True
4,16302463200535040,FFI,31.0,475.199790,True,True
...,...,...,...,...,...,...
8950,6734485286788609664,SPOC,93.0,120.000000,True,True
8951,6759481141756109056,FFI,13.0,1425.599392,True,True
8952,6759481141756109056,FFI,27.0,475.199782,True,True
8953,6759481141756109056,FFI,92.0,158.399926,True,True


Saved in tess_id_sector_exptime_G16d20_all_filtered_UPDATED_SEC.csv
